In [ ]:
#!/usr/bin/env python3"""Run BankLoanSta pipeline."""import sysimport ossys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))import pandas as pdimport numpy as npimport optunaimport kagglehubfrom core.GenericDataPipeline import GenericDataPipelinefrom core.seed_utils import SEED, set_all_seedspd.set_option('future.infer_string', False)set_all_seeds()pipeline = GenericDataPipeline()path = kagglehub.dataset_download("zaurbegiev/my-dataset")csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]csv_path = os.path.join(path, csv_files[1])df = pd.read_csv(csv_path)df = df.dropna(subset=['Loan Status']).reset_index(drop=True)df = df.drop(columns=['Loan ID', 'Customer ID'])df = pipeline.preprocessing(df)label = "Loan Status"print(f"Dataset shape: {df.shape}")print(f"Target: {label}")print(f"Target distribution:\n{df[label].value_counts()}")X = df.drop(label, axis=1)y = df[label]feature_scores = pipeline.rank_features(X, y)features_with_nulls = feature_scores[feature_scores['null_ratio'] > 0.01]# --- AXGB Feature Comparison ---print(f"\n{'='*120}")print("AXGB FEATURE COMPARISON")print(f"{'='*120}")from axgb.axgb_feature_comparison import AXGBFeatureComparison# Define extended features (fill in the list with feature names)extended_features = []  # TODO: Add extended feature names hereprint(f"Extended features: {extended_features}")print(f"Running AXGB comparison...")# Initialize and run AXGB comparisonaxgb_comparison = AXGBFeatureComparison(    df=df.copy(),    features_extended=extended_features,    target_col=label,    n_estimators=30,    learning_rate=0.3,    max_depth=6,    max_window_size=1000,    min_window_size=None,    detect_drift=False,    update_strategy='replace')# Run the comparisonresult = axgb_comparison.run()print(f"\n{'='*120}")print("AXGB SUMMARY")print(f"{'='*120}")print(f"AUC (Basic features only):  {result['auc_without_extended']:.4f}")print(f"AUC (All features):         {result['auc_with_all']:.4f}")print(f"Difference:                 {result['difference']:+.4f}")print(f"{'='*120}")print("\nAXGB Analysis Complete!")